# Data Split

This notebook pulls the data directly from the [Github repository assocaited with Jablonka et al. *Nature Communications* 2021](https://github.com/byooooo/dispersant_screening_PAL), splits the data into 5 categories based on its deltaGmin value and then preforms a stratified 5 fold split. Resulting data is saved in `data/disp_dataset.csv` and splits are saved in `data/data_split.csv`.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error as mse
import GPy
import shap
import pickle
import functools
from IPython.display import display
from IPython.display import Image

 /Users/dja3/envir/ii/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning:IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


## Get Data

In [2]:
feature_url = "https://raw.githubusercontent.com/byooooo/dispersant_screening_PAL/refs/heads/master/data/new_features_full_random.csv"
deltaG_url = "https://raw.githubusercontent.com/byooooo/dispersant_screening_PAL/refs/heads/master/data/b1-b21_random_deltaG.csv"

features = [
    "num_[W]",
    "max_[W]",
    "num_[Tr]",
    "max_[Tr]",
    "num_[Ta]",
    "max_[Ta]",
    "num_[R]",
    "max_[R]",
    "[W]",
    "[Tr]",
    "[Ta]",
    "[R]",
    "rel_shannon",
    "length",
]
dfx = pd.read_csv(feature_url)[features]
dfy = pd.read_csv(deltaG_url, usecols=[0])

df = pd.concat([dfx, dfy], axis=1)

df.sort_values(by="deltaGmin", inplace=True, ignore_index=True)
df.to_csv("data/disp_dataset.csv", index=False)

In [3]:
df.head()

,num_[W],max_[W],num_[Tr],max_[Tr],num_[Ta],max_[Ta],num_[R],max_[R],[W],[Tr],[Ta],[R],rel_shannon,length,deltaGmin
0,0.000000,0,0.400,2,0.200000,2,0.400000,5,0.100000,0.300000,0.300000,0.300000,0.356161,40,-20.673496
1,0.000000,0,0.375,2,0.375000,2,0.250000,3,0.105263,0.263158,0.315789,0.315789,0.361861,38,-20.286739
2,0.200000,2,0.200,2,0.400000,4,0.200000,5,0.117647,0.176471,0.352941,0.352941,0.366673,34,-20.253473
3,0.125000,2,0.250,2,0.250000,6,0.375000,4,0.111111,0.222222,0.333333,0.333333,0.365781,36,-19.970056
4,0.142857,2,0.000,0,0.571429,3,0.285714,3,0.125000,0.125000,0.375000,0.375000,0.362256,32,-19.821176


Assign a class to each datapoint based on its y value

In [4]:
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values * (-1)
c = (df.index.values * 5 / len(y)).astype(int)

## Split the data

Warning: Uncommenting last line and running this block will overwrite the splits used in the manuscript.

In [5]:
skf = StratifiedKFold(n_splits=5, shuffle=True)
skf.get_n_splits()

index_df = pd.DataFrame(index=range(len(y)), columns=range(5))

for i, (train_index, test_index) in enumerate(skf.split(X, c)):

    # Get training data and scale
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X[train_index, :])
    X_test = scaler.transform(X[test_index, :])
    y_train = y[train_index].reshape(-1, 1)
    y_test = y[test_index].reshape(-1, 1)

    # Save the indices
    index_df.iloc[:, i] = np.isin(np.arange(len(y)), train_index)

# Uncomment following line to overwrite splits
# index_df.to_csv("data/data_split.csv", index=False)